In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install PyPDF2 nltk sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.6 MB/s eta 0:00:00


In [3]:
import PyPDF2

def load_pdf(file_path):
    text = ""
    with open(file_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text

raw_text = load_pdf("/content/drive/MyDrive/Vunani Employee Handbook.pdf")

In [51]:
import re

def clean_text(text):
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

cleaned_text = clean_text(raw_text)

In [52]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

def sentence_chunking(text, chunk_size=5):
    sentences = sent_tokenize(text)
    chunks = []

    for i in range(0, len(sentences), chunk_size):
        chunk = " ".join(sentences[i:i+chunk_size])
        chunks.append(chunk)

    return chunks

sentence_chunks = sentence_chunking(cleaned_text)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [53]:
def paragraph_chunking(text):
    paragraphs = text.split("\n")
    return [p.strip() for p in paragraphs if len(p.strip()) > 50]

paragraph_chunks = paragraph_chunking(raw_text)

In [54]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [55]:
chunks = paragraph_chunks  # choose best strategy

embeddings = embedding_model.encode(chunks)

In [56]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [57]:
def retrieve(query, k=3):
    query_embedding = embedding_model.encode([query])[0]

    similarities = [
        cosine_similarity(query_embedding, emb)
        for emb in embeddings
    ]

    top_k_idx = np.argsort(similarities)[-k:][::-1]

    results = []
    for idx in top_k_idx:
        results.append({
            "chunk": chunks[idx],
            "score": similarities[idx]
        })

    return results

In [58]:
!pip install transformers accelerate torch

In [59]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [60]:
def build_prompt(query, retrieved_chunks):
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
You are an HR policy assistant. Answer the following question based ONLY on the provided context. Do not repeat the question or any part of the prompt in your answer.

Context:
{context}

Question: {query}
Answer:"""
    return prompt

In [74]:
def generate_answer_rag(query):
    retrieved_data = retrieve(query)

    context = "\n\n".join([item["chunk"] for item in retrieved_data])

    prompt = f"""
You are an HR assistant. Answer ONLY using the context below.

Context:
{context}

Question: {query}
Answer:
"""

    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.2
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("Answer:")[-1].strip()

    # Confidence score (average similarity) - though not returned, good to keep for potential use
    # scores = [item["score"] for item in retrieved_data]
    # confidence = np.mean(scores)

    return retrieved_data, response

In [75]:
def interpret_confidence(score):
    if score > 0.7:
        return "🟢 High Confidence"
    elif score > 0.5:
        return "🟡 Medium Confidence"
    else:
        return "🔴 Low Confidence"

In [76]:
query = "What is the policy on working another job while employed?"

retrieved, rag_answer = generate_answer_rag(query)

print("Retrieved Context:\n")
for i, chunk in enumerate(retrieved):
    print(f"\n--- Chunk {i+1} ---\n{chunk['chunk']}")

print("\n RAG Answer:\n", rag_answer)

Retrieved Context:


--- Chunk 1 ---
The employee will not be required to work on these days and the Company complies with the law that

--- Chunk 2 ---
13.3 External Employment or Contract Work and/or Declaration of Conflict of Interests

--- Chunk 3 ---
For the company’s detailed Health and Safety Policy please refer to policies under the Employee Self

 RAG Answer:
 The policy is that the employee is not required to work on these days and the Company complies with the law
